# Autoresearch Quick Start on RunPod

This notebook gets you from zero to your first training result on a RunPod GPU pod.

**Prerequisites:**
- A RunPod GPU pod with JupyterLab (port 8888)
- An Anthropic API key for the autonomous agent
- `/workspace` directory available (standard on RunPod pods)

**What this notebook does:**
1. Verifies your GPU environment
2. Clones the autoresearch-unified repo and installs dependencies
3. Downloads and prepares training data
4. Runs a baseline training pass

In [ ]:
# Environment Check
import torch
import os
import shutil

print("=== GPU Info ===")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  VRAM:   {vram_gb:.1f} GB")
    print(f"  CUDA:   {torch.version.cuda}")
else:
    print("  WARNING: No GPU detected! Training will be very slow.")

print("\n=== System Info ===")
print(f"  PyTorch: {torch.__version__}")
print(f"  /workspace exists: {os.path.isdir('/workspace')}")
total, used, free = shutil.disk_usage('/workspace')
print(f"  Disk: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")

In [ ]:
# Clone/update repo and install dependencies
import subprocess, os

repo_dir = "/workspace/autoresearch-unified"
if not os.path.exists(repo_dir):
    print("Cloning autoresearch-unified...")
    subprocess.run(["git", "clone", "https://github.com/elementalcollision/autoresearch-unified.git", repo_dir], check=True)
else:
    print("Updating existing repo...")
    subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

print("\nInstalling dependencies (this may take a few minutes)...")
subprocess.run(["pip", "install", "-e", ".[all-cuda]"], cwd=repo_dir, check=True)
print("\nInstallation complete.")

In [ ]:
# Configuration
import os, getpass

os.environ["AUTORESEARCH_BACKEND"] = "cuda"
os.environ["AUTORESEARCH_CACHE_DIR"] = "/workspace/.cache/autoresearch"
os.environ["PYTHONPATH"] = "/workspace/autoresearch-unified"

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter Anthropic API key: ")

print("Configuration set.")

In [ ]:
# Data Preparation (~3-5 minutes)
import subprocess, sys, os

os.chdir("/workspace/autoresearch-unified")
print("Downloading data and training tokenizer...")

proc = subprocess.Popen(
    [sys.executable, "prepare.py", "--num-shards=10"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nData preparation {'complete' if proc.returncode == 0 else 'FAILED'}.")

In [ ]:
# Baseline Training (~5 minutes)
import subprocess, sys

print("Running baseline training...")
proc = subprocess.Popen(
    [sys.executable, "-u", "train.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    cwd="/workspace/autoresearch-unified"
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nTraining {'complete' if proc.returncode == 0 else 'FAILED'}.")

## Next Steps

Your baseline training is complete. Now open **`02_experiment_loop.ipynb`** to run the autonomous
Claude-driven experiment loop, which will iteratively propose hyperparameter changes, train,
evaluate, and keep or discard each result.